# M5.0d Primer: Current in Disordered Media and Percolation

Book pointer: chapter 10, section 10.4 (current distributions in disordered media).

Fourth primer of module 5. Primers 00a and 00b treated conductors as uniform and circuits as a handful of resistors. This one asks what happens when a material is a random jumble of conducting and insulating pieces, as most real materials are. The answer turns on a single geometric question, whether the conducting pieces connect from one side to the other, and that question has a sharp answer: below a critical fraction nothing conducts, above it a connected path appears and the conductivity rises from zero. The notebook builds the ideas on small lattices you can solve by hand, then uses `emlib.percolation` to measure the threshold, the conductivity curve and the way current concentrates in a few bonds. It prepares you for notebook `04_percolation`, which does the same measurements systematically.

## 1. Warm-up

Try these on paper before running anything. Answers are in section 7.

1. A powder of copper grains and plastic grains is pressed into a pellet. Why does its conductivity depend on more than the volume fraction of copper?
2. A one-dimensional chain of $L$ bonds, each present with probability $p < 1$. What is the probability that the whole chain conducts? What happens as $L \to \infty$?
3. What is a cluster in a random network, and what is the spanning cluster?
4. Why is the threshold for bond percolation on the square lattice exactly $p_c = 1/2$?
5. Does every bond of the spanning cluster carry current? If not, which ones carry none?
6. Effective-medium theory predicts $\sigma_{\text{eff}} = 2(p - \tfrac12)$ for the square bond lattice. Where should it be accurate, and where should it fail?
7. Why do averages over random lattices need many samples, and how does the number needed change with the size of the lattice?

## 2. The ideas

**Real materials are not uniform.** Primer 00a treated a conductor as a continuum with one conductivity $\sigma$. Many materials of practical interest are mixtures instead. A conducting composite is a polymer filled with carbon black or metal flakes. A sedimentary rock conducts through brine in its pore space while the mineral grains are insulators. A thin metal film deposited on glass starts as isolated islands and becomes a continuous sheet only as more metal arrives. A cracked electrode, a corroded wire, a battery electrode made of particles: each is a random arrangement of conducting and insulating regions on some small scale, and the question is what conductivity it shows on the large scale.

**A random mixture is a random resistor network.** Primer 00b showed that any conducting body, cut into small cells, becomes a network of resistors joining neighbouring cells (that is how `emlib.circuits.solve_sheet` works). A mixture is the same network with some resistors missing: where two neighbouring cells are both conducting the link has a conductance, where either is insulating it has none. The simplest model keeps only that: a square lattice of nodes, each bond between neighbours present with probability $p$ and absent otherwise, all present bonds equal. This is *bond percolation*. Kirchhoff's laws on the surviving bonds give the potential at every node and the current in every bond, exactly as for any circuit.

**Connectivity decides conduction.** If you add bonds one at a time to an empty lattice, small groups of connected bonds form, the *clusters*. Current can only flow from one electrode to the other through a cluster that touches both, a *spanning cluster*. As long as no such cluster exists the network is an insulator, however many bonds it has. The moment one appears, current flows. So the effective conductivity is first of all a question of geometry, not of the conductance of each bond.

**The threshold is sharp.** For a large lattice the change from no spanning cluster to a spanning cluster happens over a narrow range of $p$ around a critical value $p_c$, the *percolation threshold*, and for an infinite lattice the change is a true discontinuity in the probability of spanning: zero below $p_c$, one above. For bonds on the square lattice $p_c = 1/2$ exactly, by a symmetry argument given in section 3.2. Other lattices and other rules (occupying sites instead of bonds, three dimensions instead of two) have other thresholds, but every one shows the same kind of sharp transition.

**Just above the threshold the network is fragile and full of dead ends.** At $p$ slightly above $p_c$ the spanning cluster is a sparse, ragged object. Much of it consists of branches that connect to the rest at a single point. No current can flow through such a branch, because current entering it has no way out: these are the *dead ends*. The bonds that do carry current form the *backbone*. Along the backbone there are single bonds through which a large share of the current must pass, and the current is spread very unevenly: a few percent of the bonds dissipate half of the power. That is why such materials heat and fail at hot spots.

**How the conductivity behaves.** Far above the threshold, a missing bond is a small defect in a nearly uniform network, and a simple self-consistent argument, the effective-medium approximation, predicts $\sigma_{\text{eff}} = 2(p - \tfrac12)$ in units of the bond conductance, which agrees with simulation to within about 0.03 for $p$ above about 0.75. Near the threshold the network is anything but nearly uniform and the conductivity follows a power law, $\sigma_{\text{eff}} \propto (p - p_c)^t$ with $t \approx 1.3$ in two dimensions, rising from zero more slowly than any straight line.

**Finite lattices and averaging.** A computer handles a finite lattice of $n \times n$ nodes. On such a lattice the transition is smeared over a width that shrinks as $n$ grows, each random realisation gives a different conductivity, and only an average over many realisations is a stable number. Knowing how big the spread is and how fast it shrinks with $n$ is part of doing the calculation honestly.

**What notebook 04 does with this.** It generates random lattices with `emlib.percolation.random_bonds`, tests spanning with `spans`, solves Kirchhoff's laws with `solve_lattice`, normalises the result with `effective_conductivity`, and marks the current-carrying bonds with `backbone`. It sweeps $p$ through the threshold for several lattice sizes, compares with the effective-medium line, and maps the current in a lattice at threshold. Every one of those steps appears below on a smaller scale.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sps
from matplotlib.collections import LineCollection
from scipy.sparse.csgraph import connected_components
from emlib import circuits as ci
from emlib import percolation as pc

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)

### 3.1 Random networks as models

The lattice used throughout has $n \times n$ nodes. Node $(i, j)$ sits at $x = i$, $y = j$. A *horizontal* bond joins $(i, j)$ to $(i+1, j)$ and a *vertical* bond joins $(i, j)$ to $(i, j+1)$, so there are $(n-1)n$ of each. Every bond is present with probability $p$, independently of all others, and a present bond has unit conductance. The left column $i = 0$ is an electrode held at $V = 1$ and the right column $i = n-1$ an electrode at $V = 0$: think of two metal bars pressed against the edges of a sample. The top and bottom edges are left open, so no current crosses them. In `emlib.percolation`, `random_bonds(n, p)` returns two boolean arrays `h_bonds` of shape $(n-1, n)$ and `v_bonds` of shape $(n, n-1)$, `True` where a bond is present.

Vertical bonds inside the two electrode columns join nodes at the same potential, so they never carry current. They are part of the lattice only so that every column looks alike.

This is *bond* percolation. The other classic model is *site* percolation, where each node is occupied with probability $p$ and neighbouring occupied nodes are connected; it has a different threshold (about 0.593 on the square lattice) but the same physics.

The conductance of a lattice is normalised so that the result is a conductivity independent of size. The full lattice is $n$ parallel rows of $n - 1$ unit bonds in series, conductance $n/(n-1)$ between the electrodes. `effective_conductivity` multiplies the measured conductance by $(n-1)/n$, so the full lattice gives exactly 1 and a lattice with no spanning path gives 0.

**Worked example 1 (a 3 by 3 lattice by hand).** Take $n = 3$, so the electrodes are the columns $i = 0$ and $i = 2$, and only the middle column of nodes $(1, 0), (1, 1), (1, 2)$ is free. Choose these bonds: horizontal $(0,0){-}(1,0)$, $(1,1){-}(2,1)$, $(0,2){-}(1,2)$, $(1,2){-}(2,2)$, vertical $(1,0){-}(1,1)$ in the middle column and $(0,0){-}(0,1)$ inside the left electrode. Everything else is absent.

Trace the paths. The top row has both of its horizontal bonds, so it is two unit resistors in series between 1 V and 0 V: current $\tfrac12$, and node $(1,2)$ sits at $V = \tfrac12$. The bottom left node $(1,0)$ connects to the left electrode through one bond, then up to $(1,1)$ through the vertical bond, then to the right electrode through one more bond: three in series, current $\tfrac13$, potentials $V_{1,0} = 1 - \tfrac13 = \tfrac23$ and $V_{1,1} = \tfrac13$. The bond inside the left electrode joins two nodes at 1 V and carries nothing. Total current $\tfrac12 + \tfrac13 = \tfrac56$, conductance $\tfrac56$ (voltage 1), effective conductivity $\tfrac56\cdot\tfrac{2}{3} = \tfrac59 = 0.556$.

Check Kirchhoff at node $(1,0)$: in from the left $\tfrac13$, out upward $\tfrac13$. At $(1,1)$: in from below $\tfrac13$, out to the right $\tfrac13$. Both balance.

**Worked example 2 (how likely is a given pattern?).** The 3 by 3 lattice has $2\cdot2\cdot3 = 12$ bonds. A particular pattern with $k$ present bonds has probability $p^k(1-p)^{12-k}$, so at $p = \tfrac12$ all $2^{12} = 4096$ patterns are equally likely, each with probability $1/4096$. The pattern of worked example 1 has 6 bonds. The simplest spanning path, one straight row of $n - 1$ bonds, appears in a given row with probability $p^{n-1}$: for $n = 64$ and $p = 0.6$ that is $0.6^{63} = 1.1\times10^{-14}$. Straight rows are therefore irrelevant on large lattices; spanning paths that exist are long, winding ones, and there are astronomically many candidates. This is why the two-dimensional threshold is $\tfrac12$ and not close to 1 as a straight-path estimate would suggest.

The code solves worked example 1 with `pc.solve_lattice` and draws the lattice with each bond coloured by its current.

In [ ]:
h = np.zeros((2, 3), dtype=bool)                  # horizontal bonds (i, j) -> (i+1, j)
v = np.zeros((3, 2), dtype=bool)                  # vertical bonds (i, j) -> (i, j+1)
h[0, 0] = h[1, 1] = h[0, 2] = h[1, 2] = True
v[1, 0] = True                                    # middle column, (1,0) -> (1,1)
v[0, 0] = True                                    # inside the left electrode
V, Ih, Iv, I = pc.solve_lattice(h, v)
print("potentials V[i, j] (rows are i = x index):\n", V)
print("horizontal bond currents (along +x):\n", Ih)
print("vertical bond currents (along +y):\n", Iv)
print(f"total current {I:.4f} (hand 5/6 = {5/6:.4f}),  effective conductivity {pc.effective_conductivity(h, v):.4f} (hand 5/9 = {5/9:.4f})")
print(f"Worked example 2: a straight row of 63 bonds at p = 0.6 is present with probability {0.6**63:.1e}")

def bond_segments(bonds, di, dj):
    i, j = np.nonzero(bonds)
    return np.stack([np.stack([i, j], axis=-1), np.stack([i + di, j + dj], axis=-1)], axis=1)

def draw_lattice(ax, h, v, Ih=None, Iv=None, lw=3, colour="0.6", electrodes=True):
    '''Present bonds as line segments; coloured by |current| if currents are given (grey where zero).'''
    segs = np.concatenate([bond_segments(h, 1, 0), bond_segments(v, 0, 1)])
    if Ih is None:
        colours = colour
    else:
        cur = np.abs(np.concatenate([Ih[h], Iv[v]]))
        colours = [colour if c < 1e-12 else plt.cm.plasma(0.15 + 0.8 * c / cur.max()) for c in cur]
    ax.add_collection(LineCollection(segs, colors=colours, linewidths=lw, capstyle="round"))
    n = h.shape[1]
    if electrodes:
        ax.plot(np.zeros(n), np.arange(n), "s", color="C3", ms=max(2, 8 - n // 8)); ax.plot(np.full(n, n - 1), np.arange(n), "s", color="C0", ms=max(2, 8 - n // 8))
    ax.set_xlim(-0.6, n - 0.4); ax.set_ylim(-0.6, n - 0.4); ax.set_aspect("equal")
    if n <= 10:
        ax.set_xticks(range(n)); ax.set_yticks(range(n)); ax.set_xlabel("i"); ax.set_ylabel("j")
    else:
        ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(figsize=(4.5, 4.5))
draw_lattice(ax, h, v, Ih, Iv, lw=5)
ax.set_title("worked example 1: colour = |current|, grey = none\nred squares: 1 V electrode, blue: 0 V"); plt.show()

### 3.2 Clusters and the threshold

A *cluster* is a set of nodes joined to each other by present bonds, with no present bond leading out of the set: a connected component of the graph of present bonds. An isolated node is a cluster of size one. A cluster that contains a node of the left column and a node of the right column is a *spanning* cluster, and the lattice *spans* if it has one. `pc.spans` finds the clusters with `scipy.sparse.csgraph.connected_components` and checks whether any cluster label appears in both electrode columns.

The quantity that shows the transition is the spanning probability $\Pi(p, n)$: the fraction of random $n \times n$ lattices at bond probability $p$ that span. For small $p$ it is essentially zero, for $p$ near 1 essentially one, and in between it rises along an S-shaped curve. As $n$ grows the curve steepens towards a step at $p_c$.

**Worked example 3 (percolation in one dimension).** One dimension can be done completely by hand and shows what a threshold is. Take an infinite chain of nodes with each bond between neighbours present with probability $p$.

*Spanning.* A stretch of $L$ bonds conducts only if all $L$ are present, probability $p^L$. For any $p < 1$ this goes to zero as $L \to \infty$: at $p = 0.99$ a chain of 100 bonds conducts with probability $0.99^{100} = 0.366$, a chain of 1000 with probability $4.3\times10^{-5}$. So the threshold in one dimension is $p_c = 1$: a single missing bond cuts the chain, and in a long enough chain there is always one.

*Cluster size.* Pick a node. The number of further nodes connected to it on the right is $k$ with probability $p^k(1-p)$ (k bonds present, then one absent), whose mean is $\sum_k k\,p^k(1-p) = p/(1-p)$. The left side gives the same. So the mean size of the cluster containing a chosen node is
$$S(p) = 1 + \frac{2p}{1-p} = \frac{1+p}{1-p}.$$
At $p = 0.5$, $S = 3$; at $p = 0.9$, $S = 19$; at $p = 0.99$, $S = 199$. The mean cluster size diverges as $p \to p_c = 1$, like $2/(1 - p)$. Diverging cluster size at the threshold is the general signature of percolation in any dimension; in two dimensions it happens at $p_c = \tfrac12$, before the lattice is full.

The code builds a chain of two million bonds and measures $S$ as the average, over nodes, of the size of the cluster each node belongs to. If $n_s$ is the number of clusters of size $s$, a randomly chosen node lies in a cluster of size $s$ with probability $s\,n_s/\sum s\,n_s$, so $S = \sum s^2 n_s / \sum s\,n_s$.

In [ ]:
rng = np.random.default_rng(0)
for p in (0.5, 0.9):
    bonds = rng.random(2_000_000) < p
    edge = np.diff(np.concatenate([[0], bonds.astype(int), [0]]))
    runs = np.nonzero(edge == -1)[0] - np.nonzero(edge == 1)[0]      # present bonds in each run
    sizes = runs + 1                                                  # nodes in each multi-node cluster
    n_isolated = len(bonds) + 1 - sizes.sum()
    sizes = np.concatenate([sizes, np.ones(n_isolated)])
    S = (sizes**2).sum() / sizes.sum()
    print(f"p = {p}: measured mean cluster size {S:.2f},  (1 + p)/(1 - p) = {(1 + p) / (1 - p):.2f}")
print(f"chains at p = 0.99: 100 bonds conduct with probability {0.99**100:.3f}, 1000 bonds with {0.99**1000:.1e}")

**Why $p_c = \tfrac12$ for bonds on the square lattice: duality.** Draw a second lattice whose nodes sit at the centres of the squares of the first, the *dual lattice*. Every bond of the original lattice is crossed by exactly one bond of the dual. Declare a dual bond present exactly when the original bond it crosses is absent. Then:

- If the original lattice has a left-to-right path of present bonds, no top-to-bottom path of dual bonds can exist, because such a path would have to cross the original path somewhere, and it can only cross at an absent original bond.
- If the original has no left-to-right path, the boundary of the cluster attached to the left electrode is a wall of absent bonds running from top to bottom, and the dual bonds crossing that wall are all present: a top-to-bottom dual path.

So exactly one of the two crossings happens, on every lattice. Now look at probabilities. Each original bond is present with probability $p$, so each dual bond is present with probability $1 - p$, and the dual of a square lattice is again a square lattice, turned through 90 degrees. Hence

$$\Pi_{\text{left-right}}(p) + \Pi_{\text{top-bottom of the dual}}(1 - p) = 1.$$

If the threshold of the square lattice is $p_c$, the dual crosses with certainty for $1 - p > p_c$ and the original crosses with certainty for $p > p_c$. Both cannot be certain at once, so $1 - p_c \ge p_c$, and both cannot fail at once, so $1 - p_c \le p_c$. Therefore $p_c = \tfrac12$.

**Worked example 4 (the dual crossing in numbers).** On a large lattice at $p = \tfrac12$ the original and dual lattices are statistically identical up to a rotation, so the two crossings are equally likely, and with exactly one of them happening each has probability $\tfrac12$. On a finite lattice of our shape the two differ slightly at the edges (the dual has one column fewer and needs to cross one row more, since the electrode columns are merged into single terminals), so the left-right crossing is slightly favoured and $\Pi(\tfrac12, n)$ is a little above $\tfrac12$, approaching it as $n$ grows. At $p = 0.4$ the dual bonds are present with probability 0.6, so on a large lattice the dual almost surely crosses and the original almost surely does not.

The code draws a small lattice with its dual and then checks the "exactly one" statement on a thousand random lattices by building the dual and testing its top-to-bottom crossing.

In [ ]:
def dual_spans(h, v):
    '''Top-to-bottom crossing of the dual lattice. Dual node (a, b) sits at x = a + 1/2, y = b - 1/2.'''
    n = h.shape[1]
    idx = lambda a, b: a * (n + 1) + b
    rows, cols = [], []
    for a in range(n - 1):                       # dual bond crossing horizontal bond (a, b)-(a+1, b)
        for b in range(n):
            if not h[a, b]:
                rows.append(idx(a, b)); cols.append(idx(a, b + 1))
    for a in range(1, n - 1):                    # dual bond crossing vertical bond (a, j)-(a, j+1); electrode columns excluded
        for j in range(n - 1):
            if not v[a, j]:
                rows.append(idx(a - 1, j + 1)); cols.append(idx(a, j + 1))
    for a in range(n - 2):                       # the bottom row and the top row are each one terminal
        rows += [idx(a, 0), idx(a, n)]; cols += [idx(a + 1, 0), idx(a + 1, n)]
    N = (n - 1) * (n + 1)
    _, lab = connected_components(sps.coo_matrix((np.ones(len(rows)), (rows, cols)), shape=(N, N)), directed=False)
    return lab[idx(0, 0)] == lab[idx(0, n)]

n_demo = 8
h, v = pc.random_bonds(n_demo, 0.5, rng=3)
fig, ax = plt.subplots(figsize=(6, 6))
draw_lattice(ax, h, v, lw=3)
dual = [((a + 0.5, b - 0.5), (a + 0.5, b + 0.5)) for (a, b), present in np.ndenumerate(h) if not present]
dual += [((a - 0.5, j + 0.5), (a + 0.5, j + 0.5)) for (a, j), present in np.ndenumerate(v) if not present and 0 < a < n_demo - 1]
ax.add_collection(LineCollection(dual, colors="C1", linewidths=2, linestyles="--"))
ax.set_xlim(-0.8, n_demo - 0.2); ax.set_ylim(-0.8, n_demo - 0.2)
ax.set_title(f"grey: present bonds, orange dashed: present dual bonds\n"
             f"left-right crossing: {pc.spans(h, v)},  dual top-bottom crossing: {dual_spans(h, v)}")
plt.show()

both = neither = 0
for s in range(1000):
    h, v = pc.random_bonds(8, 0.5, rng=s)
    a, b = pc.spans(h, v), dual_spans(h, v)
    both += a and b; neither += (not a) and (not b)
print(f"1000 random 8 x 8 lattices: both cross {both} times, neither crosses {neither} times")
assert both == 0 and neither == 0
for n in (8, 16, 32):
    Pi = np.mean([pc.spans(*pc.random_bonds(n, 0.5, rng=s + 10**6)) for s in range(2000)])
    print(f"n = {n:2d}: spanning probability at p = 1/2 is {Pi:.3f}  (sampling error about 0.011)")

The spanning probability for three lattice sizes, 200 random lattices per point. The curves cross close to $p = \tfrac12$, and the larger the lattice the steeper the rise: on an infinite lattice the curve becomes a step from 0 to 1 at $p_c$. The width of the rise is the range of $p$ over which a finite lattice cannot tell whether it is above or below the threshold. Section 3.5 measures how it shrinks with $n$.

In [ ]:
ps_span = np.linspace(0.30, 0.70, 17)
span_curves = {}
for n in (16, 32, 64):
    span_curves[n] = np.array([np.mean([pc.spans(*pc.random_bonds(n, p, rng=5000 * n + 37 * k + s)) for s in range(200)])
                               for k, p in enumerate(ps_span)])
    plt.plot(ps_span, span_curves[n], "o-", ms=4, label=f"n = {n}")
plt.axvline(pc.P_C_BOND_SQUARE, color="grey", ls=":", label="p_c = 1/2")
plt.xlabel("bond probability p"); plt.ylabel("fraction of lattices that span"); plt.legend(); plt.title("spanning probability"); plt.show()
for n, P in span_curves.items():
    print(f"n = {n:2d}: P(0.40) = {P[4]:.2f}, P(0.45) = {P[6]:.2f}, P(0.50) = {P[8]:.2f}, P(0.55) = {P[10]:.2f}, P(0.60) = {P[12]:.2f}")

### 3.3 Conductivity of a random network

Spanning is a yes-or-no property; the effective conductivity measures how well a spanning lattice conducts. Below $p_c$ it is zero on an infinite lattice. Above $p_c$ it rises, reaching 1 at $p = 1$. Two questions: what is the shape of the curve, and can it be predicted?

**Worked example 5 (the resistance between neighbouring nodes of an infinite lattice).** The effective-medium argument below needs one fact about a uniform square lattice of bonds with conductance $g$: the conductance between two neighbouring nodes $a$ and $b$, measured through the whole infinite lattice, is $2g$. Superposition gives it in three lines. Inject current $I$ at $a$ and take it out at infinity. By symmetry it splits equally into the four bonds at $a$, so the bond $a{-}b$ carries $I/4$ from $a$ to $b$. Separately, inject $I$ at infinity and take it out at $b$: again the bond $a{-}b$ carries $I/4$ from $a$ to $b$. Add the two: $I$ enters at $a$ and leaves at $b$, and the bond $a{-}b$ carries $I/2$. Its voltage is then $V_{ab} = (I/2)/g$, so the conductance of the whole lattice between $a$ and $b$ is $I/V_{ab} = 2g$. The bond $a{-}b$ itself contributes $g$ of that, so the *rest* of the lattice, seen from the two ends of one bond, has conductance $G' = g$. In general a lattice where every node has $z$ bonds gives $G' = (z/2 - 1)g$; the square lattice has $z = 4$.

The code checks it on a $101 \times 101$ lattice with its outer boundary grounded: inject $+1$ at the centre node and $-1$ at its neighbour and read off the voltage between them.

In [ ]:
n = 101
he, ve = ci.grid_edges(n, n)
edges = np.concatenate([he.reshape(-1, 2), ve.reshape(-1, 2)])
fixed = np.zeros((n, n), dtype=bool); fixed[0, :] = fixed[-1, :] = fixed[:, 0] = fixed[:, -1] = True
a, b = 50 * n + 50, 51 * n + 50
inj = np.zeros(n * n); inj[a], inj[b] = 1.0, -1.0
Vn, _ = ci.solve_network(n * n, edges, np.ones(len(edges)), fixed.ravel(), np.zeros(n * n), inj)
print(f"resistance between neighbouring nodes: {Vn[a] - Vn[b]:.5f}   (infinite lattice: 1/2)")

**Worked example 6 (the effective-medium approximation).** Replace the random network by a uniform one in which every bond has the same conductance $g_m$, chosen so that the uniform network carries the same current as the random one on average. To find $g_m$, take the uniform network with a uniform field applied, so every bond along the field has voltage $V_m$ across it, and replace one bond by a bond of conductance $g$ drawn from the random distribution ($g = 1$ with probability $p$, $g = 0$ with probability $1 - p$).

Seen from the two ends of that bond, the rest of the network is a Thevenin source: an open-circuit voltage $V_0$ behind a conductance $G' = g_m$ (worked example 5). With the original bond $g_m$ in place the voltage across it is $V_m = V_0 G'/(G' + g_m)$; with $g$ in place it is $V = V_0 G'/(G' + g)$. Their ratio gives the change caused by the substitution:
$$\delta V = V - V_m = V_m\,\frac{g_m - g}{g + g_m}.$$
A good choice of $g_m$ is one for which these disturbances cancel on average: $\langle\delta V\rangle = 0$. With the two possible values of $g$,
$$p\,\frac{g_m - 1}{1 + g_m} + (1-p)\,\frac{g_m - 0}{0 + g_m} = 0 \quad\Longrightarrow\quad p(g_m - 1) + (1-p)(1 + g_m) = 0 \quad\Longrightarrow\quad g_m = 2p - 1.$$
So $\sigma_{\text{eff}} = 2(p - \tfrac12)$ in units of the bond conductance, and zero for $p \le \tfrac12$. It predicts the threshold exactly (for this lattice) and a straight line above it: $\sigma_{\text{eff}} = 0.1$ at $p = 0.55$, $0.2$ at $0.6$, $0.5$ at $0.75$, $0.8$ at $0.9$.

**Worked example 7 (why the straight line fails near $p_c$).** Effective-medium theory treats each bond as sitting in an average environment. Near the threshold the environment is nothing like average: the spanning cluster is a tenuous, branching object with holes of all sizes, and whether a bond matters depends on whether it lies on one of the few paths across. The correct behaviour close to $p_c$ is a power law,
$$\sigma_{\text{eff}} \approx A\,(p - p_c)^t, \qquad t \approx 1.30 \text{ in two dimensions}.$$
Compare the two predictions when the distance to the threshold is halved, from $p - p_c = 0.1$ to $0.05$. The straight line drops by a factor of 2. The power law drops by $2^{1.3} = 2.46$. Closer still the gap widens: halving three times gives $2^3 = 8$ against $2^{3.9} = 14.9$. The simulated curve below falls under the straight line near the threshold for exactly this reason, and on a finite lattice it also does not quite reach zero at $p_c$, because a finite lattice sometimes spans below the threshold.

The code averages `effective_conductivity` over 30 lattices of $40 \times 40$ nodes at each $p$ and plots the mean with its spread.

In [ ]:
n_sweep, samples = 40, 30
ps_sweep = np.arange(0.40, 0.951, 0.05)
sigma_runs = np.array([[pc.effective_conductivity(*pc.random_bonds(n_sweep, p, rng=s + int(p * 1e4))) for s in range(samples)]
                       for p in ps_sweep])
sigma_mean, sigma_std = sigma_runs.mean(axis=1), sigma_runs.std(axis=1)
ema = np.maximum(0.0, 2 * (ps_sweep - 0.5))
plt.errorbar(ps_sweep, sigma_mean, yerr=sigma_std, fmt="o", ms=4, capsize=3, label=f"simulation, {n_sweep} x {n_sweep}, {samples} lattices")
pp = np.linspace(0.4, 1.0, 200)
plt.plot(pp, np.maximum(0, 2 * (pp - 0.5)), "k--", label="effective medium 2(p - 1/2)")
plt.axvline(0.5, color="grey", ls=":")
plt.xlabel("bond probability p"); plt.ylabel("effective conductivity (bond units)"); plt.legend(); plt.title("conductivity of a random bond lattice"); plt.show()
print("   p     simulated    effective medium")
for p, m, e in zip(ps_sweep, sigma_mean, ema):
    print(f"{p:5.2f}   {m:8.3f}     {e:8.3f}")

### 3.4 The backbone and the current distribution

A spanning cluster is not the same as the set of bonds that carry current. Picture a branch of the spanning cluster attached to the rest at a single node. Current entering the branch at that node has no second exit, so by Kirchhoff's current law the net current into the branch is zero, and because the branch contains no sources every potential inside it settles to the potential of the attachment node: no current flows anywhere in the branch. Such branches are *dead ends*. The bonds that remain, those that lie on at least one path from electrode to electrode that does not revisit a node, form the *backbone*. `pc.backbone` finds them numerically, as the bonds whose current is not zero to rounding error.

Within the backbone the current is spread very unevenly. Some bonds are *red bonds*: cutting any one of them would break every path across the sample, so all the current through that part of the network passes through them. Others sit in loops that carry a small share. At the threshold the bond currents range over several decades, and a small fraction of the bonds carries most of the current and dissipates most of the power. A material made this way fails at its red bonds first: they heat, oxidise, crack or melt, which removes them and pushes their current onto the next weakest path.

How big is the backbone? At the threshold of an infinite lattice the spanning cluster is a fractal of dimension about 1.90 (the number of its bonds in a window of size $L$ grows like $L^{1.90}$) and the backbone a fractal of dimension about 1.64, so the backbone is a vanishing fraction of the cluster on large scales. On a lattice of 64 by 64 that limit is still far away, and about half the bonds of the spanning cluster carry current; the rest are dead ends, and the bonds not in the spanning cluster at all carry nothing either.

**Worked example 8 (a hot spot).** Apply 1 V across a full $64 \times 64$ lattice. Each row is 63 unit bonds in series, so each bond carries $1/63 = 0.0159$ and the total current is $64/63 = 1.016$. Now take a lattice at $p = 0.5$ that just spans. Its conductivity is a few percent of the full lattice's, so the total current is a few percent of 1.016, yet the code finds a single bond carrying about two thirds of that total: about $0.024$, more than the $0.0159$ every bond carries in the full lattice. The power in a bond is $I_b^2$ (unit conductance), so that one bond dissipates more than twice the power of any bond in the full lattice, while the sample as a whole conducts 27 times less. Uneven current, not total current, decides where a disordered conductor fails.

**Animation.** A lattice between two electrodes fills with bonds as $p$ rises from 0.3 to 0.7. Clusters grow and merge but nothing flows until, close to $p = \tfrac12$, one cluster touches both electrodes. From then on the bonds are coloured by the current they carry: the backbone lights up, the dead ends stay grey, and the readout of $\sigma_{\text{eff}}$ rises from zero as more parallel paths join.

In [ ]:
from emlib import show
show.animation("m5_circuits", "PercolationBackbone")

In [ ]:
n = 64
h, v = pc.random_bonds(n, 0.5, rng=0)
assert pc.spans(h, v)
V, Ih, Iv, I = pc.solve_lattice(h, v)
bh, bv = pc.backbone(Ih, Iv)

def spanning_cluster_bonds(h, v):
    '''Mask of present bonds (horizontal, vertical) in a spanning cluster. Vertical bonds inside the electrodes are left out.'''
    n = h.shape[1]
    vi = v.copy(); vi[0, :] = vi[-1, :] = False
    he, ve = ci.grid_edges(n, n)
    edges = np.concatenate([he.reshape(-1, 2), ve.reshape(-1, 2)])
    present = np.concatenate([h.ravel(), vi.ravel()])
    adj = sps.coo_matrix((np.ones(present.sum()), (edges[present, 0], edges[present, 1])), shape=(n * n, n * n))
    _, lab = connected_components(adj, directed=False)
    span_labels = np.intersect1d(lab.reshape(n, n)[0, :], lab.reshape(n, n)[-1, :])
    in_span = np.isin(lab[edges[:, 0]], span_labels) & present
    return in_span[: h.size].reshape(h.shape), in_span[h.size :].reshape(v.shape)

sh, sv = spanning_cluster_bonds(h, v)
n_present = h.sum() + v[1:-1, :].sum()
n_span, n_back = sh.sum() + sv.sum(), bh.sum() + bv.sum()
sigma = I * (n - 1) / n
cur = np.abs(np.concatenate([Ih[bh], Iv[bv]]))
c = np.sort(cur)[::-1]
share = np.cumsum(c**2) / np.sum(c**2)
k_half = np.searchsorted(share, 0.5) + 1
print(f"present bonds {n_present},  in the spanning cluster {n_span},  carrying current (backbone) {n_back}")
print(f"effective conductivity {sigma:.4f},  total current {I:.4f}")
print(f"bond currents range over a factor {c.max() / c.min():.0f};  the largest carries {c.max() / I:.2f} of the total, i.e. {c.max():.4f}")
print(f"{k_half} bonds ({k_half / n_back:.1%} of the backbone) dissipate half of the power")
print(f"full lattice at 1 V: every bond carries 1/63 = {1 / 63:.4f}; hottest bond here / full-lattice bond: current x{c.max() * 63:.2f}, power x{(c.max() * 63)**2:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5.6))
v_inner = v.copy(); v_inner[0, :] = v_inner[-1, :] = False
draw_lattice(axes[0], h & ~sh, v_inner & ~sv, lw=0.6, colour="0.8", electrodes=False)      # other clusters
draw_lattice(axes[0], sh & ~bh, sv & ~bv, lw=1.0, colour="0.45", electrodes=False)         # dead ends of the spanning cluster
segs = np.concatenate([bond_segments(bh, 1, 0), bond_segments(bv, 0, 1)])
logc = np.log10(np.abs(np.concatenate([Ih[bh], Iv[bv]])) / c.max())
axes[0].add_collection(LineCollection(segs, colors=plt.cm.plasma(0.15 + 0.85 * (logc.clip(-4, 0) + 4) / 4), linewidths=1.8))
draw_lattice(axes[0], np.zeros_like(h), np.zeros_like(v), electrodes=True)
axes[0].set_title("p = 0.5: backbone coloured by log |current|\nspanning-cluster dead ends and other clusters grey")
axes[1].hist(np.log10(c / I), bins=40, color="C1")
axes[1].set_xlabel("log10( bond current / total current )"); axes[1].set_ylabel("number of backbone bonds"); axes[1].set_title("bond currents span several decades")
axes[2].plot(np.arange(1, len(c) + 1) / len(c), share, "C2")
axes[2].axhline(0.5, color="grey", ls=":"); axes[2].set_xscale("log")
axes[2].set_xlabel("fraction of backbone bonds, hottest first"); axes[2].set_ylabel("fraction of total power"); axes[2].set_title("a few bonds dissipate most of the power")
plt.tight_layout(); plt.show()

### 3.5 Finite size and averaging

Every number from a random lattice is one sample of a random variable. Two lattices at the same $p$ and $n$ give different conductivities, and a result is only meaningful with its spread. Three effects matter.

*Sample-to-sample fluctuations.* Well above the threshold a large lattice behaves like many independent smaller pieces put together, so its conductivity averages over them and the spread shrinks as the lattice grows. For an $n \times n$ lattice the number of bonds grows like $n^2$ and the standard deviation of $\sigma_{\text{eff}}$ falls roughly like $1/n$ (one over the square root of the number of bonds). Near the threshold this self-averaging fails: a single red bond can decide whether the lattice conducts at all, and the spread stays large.

*The standard error of a mean.* If $N$ independent samples have standard deviation $s$, their mean has standard error $s/\sqrt N$. Quoting a mean to a given precision therefore sets the number of samples.

*Shift and width of the transition.* On a finite lattice the spanning probability rises over a range of $p$ whose width shrinks like $n^{-1/\nu}$, with $\nu = 4/3$ in two dimensions, so like $n^{-3/4}$. Going from $n = 16$ to $n = 64$ narrows it by $4^{3/4} = 2.83$. The value of $p$ at which half the lattices span drifts towards $p_c$ as $n$ grows.

A last quantity describes how much of the lattice belongs to the spanning cluster: the fraction $P_\infty(p)$ of all bond positions occupied by a bond of the spanning cluster. It is zero below the threshold and rises steeply above it, like $(p - p_c)^{\beta}$ with the small exponent $\beta = 5/36$, which is why it shoots up so quickly. It is the *order parameter* of the transition, and it tends to $p$ as $p \to 1$ when every present bond is in the one big cluster.

**Worked example 9 (how many samples?).** The code below measures the spread of $\sigma_{\text{eff}}$ at $p = 0.6$ for 25 lattices at each of three sizes. Suppose a lattice of $n = 16$ gives a standard deviation $s \approx 0.06$. To quote the mean to $\pm0.005$ you need $s/\sqrt N \le 0.005$, so $N \ge (0.06/0.005)^2 = 144$ lattices. At $n = 64$ the spread is between three and four times smaller (3.6 in this run, against 4 from the $1/n$ rule), $s \approx 0.017$, and $N \ge (0.017/0.005)^2 \approx 12$ lattices are enough. Each $64 \times 64$ lattice costs about 16 times as much to solve as a $16 \times 16$ one, so the total work is about the same; what the larger lattice buys is a smaller systematic finite-size error.

**Worked example 10 (the width of the transition).** If the spanning probability rises from 0.2 to 0.8 over a width $w(n) \propto n^{-3/4}$, then $w(64)/w(16) = 4^{-3/4} = 0.35$ and $w(32)/w(16) = 2^{-3/4} = 0.59$. The code estimates $w$ from the spanning curves of section 3.2 by interpolating where each curve crosses 0.2 and 0.8. With 200 lattices per point the estimate is noisy at the 0.01 level, so expect the trend, not the exact ratio.

In [ ]:
print("p = 0.6, 25 lattices per size:")
spread = {}
for n in (16, 32, 64):
    s = np.array([pc.effective_conductivity(*pc.random_bonds(n, 0.6, rng=k + 77 * n)) for k in range(25)])
    spread[n] = s.std(ddof=1)
    print(f"  n = {n:2d}: mean {s.mean():.4f},  standard deviation {spread[n]:.4f},  standard error of the mean {spread[n] / 5:.4f}")
print(f"  ratio of spreads n = 16 / n = 64: {spread[16] / spread[64]:.1f}  (1/n scaling predicts 4)")
for n in (16, 64):
    print(f"  samples needed for +-0.005 at n = {n}: {int(np.ceil((spread[n] / 0.005) ** 2))}")

def crossing(ps, P, level):
    return np.interp(level, np.maximum.accumulate(P), ps)

print("\nwidth of the spanning transition (p at 0.8 minus p at 0.2):")
widths = {n: crossing(ps_span, P, 0.8) - crossing(ps_span, P, 0.2) for n, P in span_curves.items()}
for n, w in widths.items():
    print(f"  n = {n:2d}: width {w:.3f},  half-spanning point p = {crossing(ps_span, span_curves[n], 0.5):.3f}")
print(f"  w(32)/w(16) = {widths[32] / widths[16]:.2f} (predicted 0.59),  w(64)/w(16) = {widths[64] / widths[16]:.2f} (predicted 0.35)")

ps_order = np.arange(0.45, 0.951, 0.05)
P_inf = []
for p in ps_order:
    frac = []
    for k in range(10):
        h, v = pc.random_bonds(64, p, rng=900 + k + int(p * 1e3))
        sh, sv = spanning_cluster_bonds(h, v)
        frac.append((sh.sum() + sv.sum()) / (h.size + v[1:-1, :].size))
    P_inf.append(np.mean(frac))
plt.plot(ps_order, P_inf, "o-", label="fraction of bond positions in the spanning cluster")
plt.plot(ps_order, ps_order, "k:", label="fraction of bond positions occupied, p")
plt.axvline(0.5, color="grey", ls=":"); plt.xlabel("p"); plt.legend(); plt.title("the order parameter, 64 x 64, 10 lattices per point"); plt.show()

### Exercise: the effective-medium prediction

Replace the body of `ema_conductivity` with your own version of the effective-medium result of worked example 6. It must accept a single $p$ or an array. The check compares it with the simulated means of section 3.3 at $p = 0.75$ and $p = 0.9$, where the effective-medium line should be within 0.03 of the simulation, and with a fresh simulation at $p = 0.3$, where every lattice must be an insulator. An insulating lattice returns a conductivity of order $10^{-16}$ rather than exactly zero: the electrode clusters sit at potentials of 1 and 0 to rounding error, and the solver reports the rounding as a tiny current.

In [ ]:
def ema_conductivity(p):
    '''Effective-medium conductivity of the square bond lattice, in units of the bond conductance.'''
    p = np.asarray(p, dtype=float)
    return np.maximum(0.0, 2.0 * (p - 0.5))          # replace with your own version


for p_check in (0.75, 0.9):
    k = int(np.argmin(np.abs(ps_sweep - p_check)))
    print(f"p = {p_check}: simulated {sigma_mean[k]:.3f},  effective medium {float(ema_conductivity(p_check)):.3f}")
    assert abs(sigma_mean[k] - ema_conductivity(p_check)) < 0.03
low = [pc.effective_conductivity(*pc.random_bonds(40, 0.3, rng=s)) for s in range(10)]
print(f"p = 0.3: largest simulated conductivity {max(low):.1e} (rounding error),  effective medium {float(ema_conductivity(0.3)):.3f}")
assert max(low) < 1e-12 and ema_conductivity(0.3) == 0.0
print("exercise passed")

## 4. Verify

Two closed-form checks. A full lattice is $n$ rows of $n - 1$ unit bonds in series, so its normalised conductivity must be exactly 1. And in any random lattice Kirchhoff's current law must hold at every node that is not on an electrode: the sum of the bond currents leaving a node is zero. The second check also confirms that the current leaving the left electrode equals the current entering the right one.

In [ ]:
n = 20
full = pc.effective_conductivity(np.ones((n - 1, n), bool), np.ones((n, n - 1), bool))
print(f"full {n} x {n} lattice: effective conductivity {full:.12f}")
assert abs(full - 1.0) < 1e-12

n = 40
h, v = pc.random_bonds(n, 0.6, rng=11)
V, Ih, Iv, I = pc.solve_lattice(h, v)
net = np.zeros((n, n))
net[:-1, :] += Ih; net[1:, :] -= Ih          # current leaving each node through its bonds
net[:, :-1] += Iv; net[:, 1:] -= Iv
print(f"largest Kirchhoff residual at an interior node: {np.abs(net[1:-1, :]).max():.1e}")
print(f"current out of the left electrode {I:.6f},  into the right electrode {Ih[-1, :].sum():.6f}")
assert np.abs(net[1:-1, :]).max() < 1e-10 and abs(I - Ih[-1, :].sum()) < 1e-12

## 5. Explore and rebuild

Explore. Measure the critical exponent. Near the threshold $\sigma_{\text{eff}} \approx A(p - p_c)^t$, so a plot of $\log\sigma_{\text{eff}}$ against $\log(p - p_c)$ should be a straight line of slope $t \approx 1.3$. Try it with the means of section 3.3 for $p$ from 0.55 to 0.7 on lattices of $n = 32$, $64$ and $128$. You will find the slope depends on $n$ and on which points you include: close to $p_c$ the finite lattice rounds the curve off, far from it the power law gives way to the effective-medium line. Find the window where the slope is stable and decide how much you trust the number. Then repeat with *site* percolation (occupy nodes with probability $p$, connect neighbouring occupied nodes) and locate its threshold, which is not $\tfrac12$.

Rebuild. Close this notebook. On paper: solve worked example 1 by hand, derive the mean cluster size of one-dimensional percolation, give the duality argument for $p_c = \tfrac12$ in four sentences, and derive the effective-medium result from the Thevenin step and the condition $\langle\delta V\rangle = 0$. In a blank editor: generate a random bond lattice, test spanning with `connected_components`, solve it with `emlib.circuits.solve_network`, and reproduce the conductivity curve of section 3.3.

## 6. Key results

| result | formula or value | holds when |
|---|---|---|
| bond percolation model | each bond present with probability $p$, unit conductance | square lattice, independent bonds |
| probability of one pattern | $p^k(1-p)^{N-k}$ for $k$ present of $N$ bonds | independent bonds |
| normalisation | $\sigma_{\text{eff}} = G\,(n-1)/n$, full lattice gives 1 | $n \times n$ nodes, electrodes on two columns |
| 1D spanning | $p^L$ for $L$ bonds; $p_c = 1$ | chain |
| 1D mean cluster size | $S = (1+p)/(1-p)$, diverges as $p \to 1$ | infinite chain |
| duality | exactly one of: left-right crossing, dual top-bottom crossing | any configuration |
| bond threshold, square lattice | $p_c = 1/2$ | infinite lattice |
| site threshold, square lattice | $p_c \approx 0.593$ | infinite lattice |
| neighbour conductance | whole lattice $2g$; rest of lattice $G' = (z/2 - 1)g = g$ | infinite uniform square lattice |
| effective medium | $\sigma_{\text{eff}} = 2(p - \tfrac12)$, zero below $\tfrac12$ | square bond lattice, within about 0.03 for $p \gtrsim 0.75$ |
| critical law | $\sigma_{\text{eff}} \propto (p - p_c)^t$, $t \approx 1.30$ | 2D, close above $p_c$ |
| order parameter | $P_\infty \propto (p - p_c)^{\beta}$, $\beta = 5/36$ | 2D, close above $p_c$ |
| fractal dimensions at $p_c$ | spanning cluster 1.90, backbone 1.64 | 2D |
| dead ends | carry no current: one attachment point, no source | any network |
| transition width | $w \propto n^{-1/\nu}$, $\nu = 4/3$ | 2D finite lattice |
| standard error | $s/\sqrt N$ for $N$ independent samples | averages over realisations |
| sample spread | $\propto 1/n$ well above $p_c$ | self-averaging regime |

## 7. Warm-up answers

1. Conduction needs a connected path of copper grains from one face to the other. The same volume fraction can be arranged as isolated grains (an insulator) or as connected chains (a conductor), so the arrangement, and in a random packing the probability of connection, matters as much as the amount.
2. All $L$ bonds must be present, probability $p^L$. For any $p < 1$ this goes to zero as $L \to \infty$, so an infinite chain never conducts unless $p = 1$: the one-dimensional threshold is $p_c = 1$.
3. A cluster is a set of nodes connected to each other by present bonds and to nothing else (a connected component). The spanning cluster is a cluster that touches both electrodes; current can flow only through it.
4. By duality: the dual lattice, with a dual bond present wherever the original is absent, is again a square lattice with bond probability $1 - p$, and exactly one of the two lattices crosses. The thresholds must satisfy $p_c = 1 - p_c$, so $p_c = 1/2$.
5. No. Dead-end branches, attached to the rest at a single node, carry no current because current entering them has no exit. Only the backbone carries current.
6. Accurate well above the threshold, where a missing bond is a small defect in a nearly uniform network (in practice within about 0.03 of simulation for $p \gtrsim 0.75$). It fails near $p_c$, where the cluster is tenuous and fractal and the conductivity follows $(p - p_c)^{1.3}$ instead of a straight line.
7. Each lattice is one random sample, and the spread between samples is large on small lattices. The standard error of a mean of $N$ samples is $s/\sqrt N$. Well above the threshold $s$ falls roughly like $1/n$, so larger lattices need fewer samples for the same precision; near the threshold the spread stays large at every size.